# Train Module 1 (Specialization) với NHIỀU ẢNH HƠN — 140 → 300

Notebook RIÊNG, chỉ tập trung train lại Module 1 với 30 ảnh/nhóm tuổi (300 ảnh, gấp ~2.1 lần
140 gốc) — không lẫn phần đánh giá FG-NET.

## Chuẩn bị trước khi chạy
1. Bật GPU (Settings > Accelerator > GPU T4).
2. Bật Internet (Settings > Internet > On) — cần để `git clone` labels đầy đủ + cài thư viện.
3. Add Input: dataset chứa checkpoint gốc VÀ **`ffhq256_images`** (70k ảnh FFHQ đầy đủ —
   BẮT BUỘC cho việc này, khác với notebook đánh giá không cần).
4. Sửa `DATA_DIR` ở cell dưới cho khớp tên dataset thật (in ra ở cell đầu tiên).

In [ ]:
import os
print("Cac dataset da gan vao notebook nay:")
for name in sorted(os.listdir("/kaggle/input")):
    print(f"  - {name}")


In [ ]:
DATA_DIR = "/kaggle/input/datasets/menonkk/nckh-2025-2026"   # <-- SUA neu ten dataset khac
OUTPUT_DIR = "/kaggle/working/FADING_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Ghi log toàn bộ session ra file — giống bản dùng ở notebook đánh giá

In [ ]:
import sys
import datetime

LOG_FILE_PATH = os.path.join(OUTPUT_DIR, "training_session_log.txt")

class TeeLogger:
    def __init__(self, filepath, mode="a"):
        self.terminal = sys.stdout
        self.log_file = open(filepath, mode, encoding="utf-8")

    def write(self, message):
        self.terminal.write(message)
        self.log_file.write(message)
        self.log_file.flush()

    def flush(self):
        self.terminal.flush()
        self.log_file.flush()

    def isatty(self):
        return self.terminal.isatty()

    def __getattr__(self, name):
        return getattr(self.terminal, name)

_true_stdout = sys.stdout
while hasattr(_true_stdout, "terminal"):
    _true_stdout = _true_stdout.terminal
sys.stdout = TeeLogger(LOG_FILE_PATH, mode="a")

print(f"\n{'='*70}")
print(f"===== BAT DAU / TIEP TUC GHI LOG - {datetime.datetime.now()} =====")
print(f"===== File log: {LOG_FILE_PATH} =====")
print(f"{'='*70}\n")


In [ ]:
!pip install -q diffusers transformers accelerate bitsandbytes peft

# SUA: nang cap torchao - peft (khi gan LoRA qua add_adapter) TU DONG kiem tra
# phien ban torchao (dung de kiem tra co dang dung ky thuat luong tu hoa torchao
# hay khong, KHONG PHAI thu vien du an dang can dung truc tiep) - ban co san tren
# Kaggle (0.10.0) qua cu so voi peft yeu cau (>0.16.0), gay ImportError cung ngay
# ca khi khong he dung tinh nang torchao. Nang cap de peft kiem tra qua duoc.
!pip install -q -U torchao


In [ ]:
import subprocess
import pandas as pd

# Tai labels DAY DU (70k anh) tu repo goc - repo nho, chi co CSV, khong can upload rieng
FULL_LABELS_REPO = "/kaggle/working/FFHQ-Aging-Dataset-full"
if not os.path.isdir(FULL_LABELS_REPO):
    subprocess.run(["git", "clone", "--depth", "1",
                     "https://github.com/royorel/FFHQ-Aging-Dataset.git", FULL_LABELS_REPO],
                    check=True)

FULL_LABELS_CSV = os.path.join(FULL_LABELS_REPO, "ffhq_aging_labels.csv")
labels_full = pd.read_csv(FULL_LABELS_CSV)
print(f"Da tai labels day du: {len(labels_full)} dong")
print(labels_full["age_group"].value_counts().sort_index())


In [ ]:
# TIM thu muc chua du 70k anh FFHQ (SUA duong dan neu ten khac trong dataset that)
FFHQ_FULL_DIR_CANDIDATES = [
    os.path.join(DATA_DIR, "ffhq256_images/ffhq256_images"),
    os.path.join(DATA_DIR, "ffhq256_images"),
]
FFHQ_FULL_DIR = next((p for p in FFHQ_FULL_DIR_CANDIDATES if os.path.isdir(p)), None)

if FFHQ_FULL_DIR is None:
    raise FileNotFoundError(
        "KHONG TIM THAY thu muc anh FFHQ day du (ffhq256_images) trong DATA_DIR. "
        "Can upload len dataset Kaggle truoc, hoac sua lai FFHQ_FULL_DIR_CANDIDATES."
    )

n_files = len(os.listdir(FFHQ_FULL_DIR))
print(f"Tim thay thu muc anh: {FFHQ_FULL_DIR} ({n_files} file)")


In [ ]:
# ===== Chon N anh MOI cho MOI nhom (10 nhom tuoi), can bang gioi tinh =====
N_PER_GROUP = 30   # goc la 14 - co the chinh lai neu muon thu gia tri khac

expanded_rows = []
for age_group in labels_full["age_group"].unique():
    group_df = labels_full[labels_full["age_group"] == age_group]
    for gender in ["male", "female"]:
        sub = group_df[group_df["gender"] == gender]
        sub_existing = sub[sub["image_number"].apply(
            lambda n: os.path.isfile(os.path.join(FFHQ_FULL_DIR, f"{int(n):05d}.png")))]
        n_take = min(N_PER_GROUP // 2, len(sub_existing))
        expanded_rows.append(sub_existing.sample(n=n_take, random_state=42))

df_expanded = pd.concat(expanded_rows, ignore_index=True)
print(f"Da chon duoc {len(df_expanded)} anh (muc tieu {N_PER_GROUP * labels_full['age_group'].nunique()})")
print(df_expanded.groupby(["age_group", "gender"]).size())

EXPANDED_LABELS_CSV = "/kaggle/working/expanded_labels.csv"
df_expanded.to_csv(EXPANDED_LABELS_CSV, index=False)
print(f"\nDa luu: {EXPANDED_LABELS_CSV}")


## Định nghĩa hàm tiện ích + class Dataset/Specializer — copy nguyên từ notebook đánh giá gốc

In [ ]:
from typing import Dict

AGE_GROUP_TO_AGE: Dict[str, int] = {
    "0-2": 1, "3-6": 4, "7-9": 8, "10-14": 12, "15-19": 17,
    "20-29": 24, "30-39": 34, "40-49": 44, "50-69": 59, "70-120": 80,
}

def age_group_to_age(age_group: str) -> int:
    return AGE_GROUP_TO_AGE[age_group]

def gender_to_word(gender: str, age: int) -> str:
    is_female = gender.lower() == "female"
    if age < 15:
        return "girl" if is_female else "boy"
    return "woman" if is_female else "man"

def build_prompt_alpha(age: int, gender_word: str) -> str:
    return f"photo of a {age} year old {gender_word}"

def build_prompt_neutral(gender_word: str) -> str:
    return f"photo of a {gender_word}"


In [ ]:
import random
from typing import List, Tuple

import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from diffusers import AutoencoderKL, DDPMScheduler, UNet2DConditionModel
from transformers import CLIPTextModel, CLIPTokenizer


class FFHQAgingDataset(Dataset):
    """Dataset doc anh FFHQ + labels CSV, tra ve (anh tensor, P_alpha, P_neutral)."""

    def __init__(self, ffhq_dir: str, labels_csv: str, image_size: int = 512):
        self.ffhq_dir = ffhq_dir
        df = pd.read_csv(labels_csv)

        self.samples: List[Tuple[str, str, str]] = []
        for _, row in df.iterrows():
            image_path = os.path.join(ffhq_dir, f"{int(row['image_number']):05d}.png")
            if not os.path.isfile(image_path):
                continue
            age = age_group_to_age(row["age_group"])
            gender_word = gender_to_word(row["gender"], age)
            self.samples.append(
                (image_path, build_prompt_alpha(age, gender_word), build_prompt_neutral(gender_word))
            )

        self.transform = transforms.Compose([
            transforms.Resize((image_size, image_size), interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
        ])

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        image_path, p_alpha, p_neutral = self.samples[idx]
        image = Image.open(image_path).convert("RGB")
        return self.transform(image), p_alpha, p_neutral


class Specializer:
    """Boc toan bo Module 1: load Stable Diffusion, chay double-prompt fine-tuning."""

    def __init__(self, pretrained_model_name_or_path: str = "runwayml/stable-diffusion-v1-5",
                 device: str = "cuda", lr: float = 2e-6, betas: Tuple[float, float] = (0.9, 0.999),
                 train_steps: int = 150, batch_size: int = 1,
                 gradient_accumulation_steps: int = 2,
                 use_lr_decay: bool = False,
                 use_warmup: bool = False, warmup_ratio: float = 0.1,
                 use_gradient_clipping: bool = False, max_grad_norm: float = 1.0,
                 use_lora: bool = False, lora_rank: int = 8):
        self.pretrained_model_name_or_path = pretrained_model_name_or_path
        self.device = device
        self.lr = lr
        self.betas = betas
        self.train_steps = train_steps
        self.batch_size = batch_size
        self.gradient_accumulation_steps = max(1, gradient_accumulation_steps)
        self.use_lr_decay = use_lr_decay
        # SUA: them Warmup - LR tang dan tu ~1% len 100% trong nhung buoc DAU (mac dinh
        # 10% tong so buoc), truoc khi (neu use_lr_decay=True) giam dan nhu cu. Ly do:
        # nhung buoc dau, UNet con "xa la" voi du lieu moi - cap nhat qua manh ngay tu
        # dau co the day trong so di sai huong truoc khi kip "lam quen".
        self.use_warmup = use_warmup
        self.warmup_ratio = warmup_ratio
        # SUA: them Gradient Clipping - gioi han do lon toi da cua gradient moi buoc,
        # tranh 1 batch "kho" (VD trung t rat lon) gay dot bien gradient, pha vo nhung
        # gi da hoc duoc o cac buoc truoc.
        self.use_gradient_clipping = use_gradient_clipping
        self.max_grad_norm = max_grad_norm
        # SUA: LoRA - CHI hoc them ~vai trieu tham so MOI (gan vao lop Attention:
        # to_k, to_q, to_v, to_out), GIU NGUYEN toan bo UNet goc (~860 trieu tham so
        # bi "dong bang"). Phu hop hon voi few-shot (it du lieu), giam rui ro "bao hoa
        # som" da thay o Loss (khong gian tim kiem nho hon nhieu).
        self.use_lora = use_lora
        self.lora_rank = lora_rank

        self.vae = None
        self.unet = None
        self.text_encoder = None
        self.tokenizer = None
        self.noise_scheduler = None
        self.optimizer = None
        self.lr_scheduler = None

    def _load_models(self) -> None:
        model_id = self.pretrained_model_name_or_path

        # SUA: vo hieu weights_only cho torch.load (PyTorch 2.6+), can cho checkpoint YOLO-style
        # o cac buoc khac - o day khong bat buoc nhung giu de dong bo an toan neu ham nay
        # duoc goi lai o dau khac trong tuong lai.
        self.vae = AutoencoderKL.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float16)
        self.text_encoder = CLIPTextModel.from_pretrained(
            model_id, subfolder="text_encoder", torch_dtype=torch.float16
        )
        self.tokenizer = CLIPTokenizer.from_pretrained(model_id, subfolder="tokenizer")
        self.unet = UNet2DConditionModel.from_pretrained(
            model_id, subfolder="unet", torch_dtype=torch.float16
        )
        self.noise_scheduler = DDPMScheduler.from_pretrained(model_id, subfolder="scheduler")

        self.vae.to(self.device).eval().requires_grad_(False)
        self.text_encoder.to(self.device).eval().requires_grad_(False)
        self.unet.to(self.device).train()
        self.unet.enable_gradient_checkpointing()

        if self.use_lora:
            from peft import LoraConfig
            lora_config = LoraConfig(
                r=self.lora_rank,
                lora_alpha=self.lora_rank,
                target_modules=["to_k", "to_q", "to_v", "to_out.0"],
                init_lora_weights="gaussian",
            )
            self.unet.add_adapter(lora_config)   # UNet goc TU DONG bi dong bang o day
            n_trainable = sum(p.numel() for p in self.unet.parameters() if p.requires_grad)
            n_total = sum(p.numel() for p in self.unet.parameters())
            print(f"[Specializer] LoRA BAT: {n_trainable:,} / {n_total:,} tham so "
                  f"co the train ({100*n_trainable/n_total:.2f}%)")

        # SUA: chi lay tham so CO THE TRAIN (LoRA-only neu bat, toan bo UNet neu khong)
        trainable_params = [p for p in self.unet.parameters() if p.requires_grad]

        # SUA (lan 2): them eps=1e-4 (thay vi mac dinh 1e-8) - eps mac dinh QUA NHO,
        # bi "chim" thanh 0 trong fp16 (UNet dang load torch_dtype=float16), gay chia
        # cho 0 trong cong thuc Adam (m / (sqrt(v)+eps)) -> NaN lan ra toan bo tham so.
        # Day CHINH XAC la loai loi da gap truoc do o Module 2 (Null-text Optimization),
        # gio quay lai o Module 1 vi vua doi tu Adam8bit (tu xu ly duoc) sang AdamW thuong.
        # 1e-4 la muc khuyen nghi chuan cho training fp16 (HuggingFace fp16 guide).
        self.optimizer = torch.optim.AdamW(trainable_params, lr=self.lr, betas=self.betas,
                                             eps=1e-4, weight_decay=1e-2)
        print("[Specializer] Dung optimizer: AdamW (eps=1e-4, an toan cho fp16)")

        # SUA: ghep Warmup (neu bat) + LR decay (neu bat) thanh 1 lich trinh NOI TIEP,
        # dung SequentialLR - Warmup chay TRUOC (tang dan), decay chay SAU (giam dan).
        if self.use_warmup or self.use_lr_decay:
            warmup_steps = max(1, int(self.warmup_ratio * self.train_steps)) if self.use_warmup else 0
            remaining_steps = max(1, self.train_steps - warmup_steps)

            phases = []
            if self.use_warmup:
                warmup_phase = torch.optim.lr_scheduler.LinearLR(
                    self.optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup_steps
                )
                phases.append(warmup_phase)

            if self.use_lr_decay:
                decay_phase = torch.optim.lr_scheduler.LinearLR(
                    self.optimizer, start_factor=1.0, end_factor=0.1, total_iters=remaining_steps
                )
            else:
                decay_phase = torch.optim.lr_scheduler.ConstantLR(
                    self.optimizer, factor=1.0, total_iters=remaining_steps
                )
            phases.append(decay_phase)

            if len(phases) > 1:
                self.lr_scheduler = torch.optim.lr_scheduler.SequentialLR(
                    self.optimizer, schedulers=phases, milestones=[warmup_steps]
                )
            else:
                self.lr_scheduler = phases[0]

    def _encode_prompts(self, prompts: List[str]) -> torch.Tensor:
        tokens = self.tokenizer(
            prompts, padding="max_length", max_length=self.tokenizer.model_max_length,
            truncation=True, return_tensors="pt",
        ).to(self.device)
        with torch.no_grad():
            return self.text_encoder(tokens.input_ids)[0]

    def train(self, dataset: FFHQAgingDataset) -> Dict[str, List[float]]:
        if self.unet is None:
            self._load_models()

        losses: List[float] = []
        lrs: List[float] = []
        grad_norms: List[float] = []
        n = len(dataset)

        total_sub_steps = self.train_steps * self.gradient_accumulation_steps
        self.optimizer.zero_grad()

        for sub_step in range(total_sub_steps):
            idxs = [random.randrange(n) for _ in range(self.batch_size)]
            images, p_alphas, p_neutrals = [], [], []
            for idx in idxs:
                image, p_alpha, p_neutral = dataset[idx]
                images.append(image)
                p_alphas.append(p_alpha)
                p_neutrals.append(p_neutral)

            images_t = torch.stack(images).to(self.device, dtype=torch.float16)

            with torch.no_grad():
                z0 = self.vae.encode(images_t).latent_dist.sample()
                z0 = z0 * self.vae.config.scaling_factor

            t = torch.randint(0, self.noise_scheduler.config.num_train_timesteps,
                              (self.batch_size,), device=self.device).long()

            eps = torch.randn_like(z0)
            eps_prime = torch.randn_like(z0)

            z_t = self.noise_scheduler.add_noise(z0, eps, t)
            z_t_prime = self.noise_scheduler.add_noise(z0, eps_prime, t)

            neutral_emb = self._encode_prompts(p_neutrals)
            alpha_emb = self._encode_prompts(p_alphas)

            latents_in = torch.cat([z_t, z_t_prime], dim=0)
            timesteps_in = torch.cat([t, t], dim=0)
            emb_in = torch.cat([neutral_emb, alpha_emb], dim=0)

            noise_pred = self.unet(latents_in, timesteps_in, encoder_hidden_states=emb_in).sample
            noise_pred_neutral, noise_pred_alpha = noise_pred.chunk(2, dim=0)

            raw_loss = F.mse_loss(noise_pred_neutral.float(), eps.float()) + F.mse_loss(
                noise_pred_alpha.float(), eps_prime.float()
            )

            # Chia nhỏ loss theo số bước tích lũy
            loss = raw_loss / self.gradient_accumulation_steps
            loss.backward()

            # Kiểm tra thời điểm cập nhật trọng số
            if (sub_step + 1) % self.gradient_accumulation_steps == 0 or (sub_step + 1) == total_sub_steps:
                grad_norm = torch.nn.utils.clip_grad_norm_(
                    self.unet.parameters(),
                    max_norm=self.max_grad_norm if self.use_gradient_clipping else float("inf"),
                )
                self.optimizer.step()
                self.optimizer.zero_grad()

                if self.lr_scheduler is not None:
                    self.lr_scheduler.step()

                current_lr = self.optimizer.param_groups[0]["lr"]
                step_idx = (sub_step + 1) // self.gradient_accumulation_steps
                losses.append(raw_loss.item())
                lrs.append(current_lr)
                grad_norms.append(float(grad_norm))

                if step_idx % 10 == 0 or step_idx == 1:
                    print(f"[Specializer] step {step_idx}/{self.train_steps} loss={raw_loss.item():.4f} "
                          f"lr={current_lr:.2e} grad_norm={float(grad_norm):.3f}")

        return {"losses": losses, "lrs": lrs, "grad_norms": grad_norms}

    def save_checkpoint(self, output_dir: str) -> None:
        os.makedirs(output_dir, exist_ok=True)
        if self.use_lora:
            # SUA: LUU RIENG phan LoRA (vai chuc MB) - KHONG luu toan bo UNet goc
            # (~1.7GB, khong doi, khong can luu lai).
            self.unet.save_lora_adapter(output_dir)
            print(f"[Specializer] da luu LoRA adapter (nho, KHONG phai UNet day du) vao {output_dir}")
        else:
            self.unet.save_pretrained(output_dir)
            print(f"[Specializer] da luu checkpoint UNet DAY DU vao {output_dir}")


In [ ]:
# Kiểm tra nhanh kích thước tensor đầu ra của FFHQAgingDataset (chuẩn 512x512)
dataset = FFHQAgingDataset(FFHQ_FULL_DIR, EXPANDED_LABELS_CSV)
sample_tensor, _, _ = dataset[0]
assert sample_tensor.shape == (3, 512, 512), f"Kích thước không đúng: {sample_tensor.shape}, kỳ vọng: (3, 512, 512)"
print(f"✓ Verification test passed: sample_tensor.shape = {sample_tensor.shape}")


## Vá lỗi weights_only (PyTorch 2.6+) — cần cho việc load YOLO-style checkpoint khác trong dự án

Không bắt buộc cho chính Module 1 (chỉ dùng CLIP/VAE/UNet chuẩn diffusers), nhưng thêm vào cho an toàn nếu code sau này cần tải checkpoint kiểu khác trong cùng phiên.

In [ ]:
if not getattr(torch.load, "_da_vo_hieu_weights_only", False):
    _torch_load_goc = torch.load
    def _torch_load_khong_weights_only(*args, **kwargs):
        kwargs.setdefault("weights_only", False)
        return _torch_load_goc(*args, **kwargs)
    _torch_load_khong_weights_only._da_vo_hieu_weights_only = True
    torch.load = _torch_load_khong_weights_only


## Chạy training — checkpoint lưu ở đường dẫn RIÊNG (`specialized_unet_v2`), KHÔNG ghi đè checkpoint cũ

Số bước train tự tính theo tỷ lệ số ảnh (giữ nguyên "mức độ nhìn thấy" mỗi ảnh so với bản gốc 140 ảnh/150 bước).

In [ ]:
import datetime

N_ORIGINAL = 140
STEPS_ORIGINAL = 150
n_actual = len(df_expanded)
train_steps_new = round(STEPS_ORIGINAL * (n_actual / N_ORIGINAL))

print(f"So anh thuc te: {n_actual} | So buoc train tinh theo ty le: {train_steps_new}")

# SUA: doi ten checkpoint (them "_adamw") de KHONG ghi de checkpoint adam8bit
# da train truoc do - giu ca 2 de so sanh song song.
CKPT_DIR_NEW = os.path.join(OUTPUT_DIR, "checkpoints", "specialized_unet_v2_adamw")

print(f"\nBat dau train Module 1 voi {n_actual} anh, {train_steps_new} buoc...")
print(f"Checkpoint se luu vao: {CKPT_DIR_NEW}\n")

dataset = FFHQAgingDataset(FFHQ_FULL_DIR, EXPANDED_LABELS_CSV)
print(f"Dataset thuc te load duoc: {len(dataset)} anh (sau loc file ton tai)\n")

specializer = Specializer(
    pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5",
    lr=2e-6,                           # Giảm từ 5e-6 xuống 2e-6 cho training 512x512
    betas=(0.9, 0.999),
    train_steps=train_steps_new,
    batch_size=1,                      # ĐỔI THÀNH 1 để giải phóng VRAM trên T4
    gradient_accumulation_steps=2,     # Tích lũy 2 bước để giữ effective batch size = 2
    use_lr_decay=True,
    use_warmup=True,
    warmup_ratio=0.1,
    use_gradient_clipping=True,
    max_grad_norm=1.0,
)
t_start = datetime.datetime.now()
train_history = specializer.train(dataset)   # SUA: gio tra ve dict {"losses","lrs","grad_norms"}
t_end = datetime.datetime.now()

losses = train_history["losses"]
lrs = train_history["lrs"]
grad_norms = train_history["grad_norms"]
max_grad_norm_used = specializer.max_grad_norm   # SUA: luu lai TRUOC KHI del specializer

specializer.save_checkpoint(CKPT_DIR_NEW)

del specializer
torch.cuda.empty_cache()

print(f"\nHoan tat training. Thoi gian: {t_end - t_start}")
print(f"Loss dau: {losses[0]:.4f} | Loss cuoi: {losses[-1]:.4f} | Loss thap nhat: {min(losses):.4f}")
print(f"LR dau: {lrs[0]:.2e} | LR cuoi: {lrs[-1]:.2e}")
n_clipped = sum(1 for g in grad_norms if g > max_grad_norm_used)
print(f"Grad norm trung binh: {sum(grad_norms)/len(grad_norms):.3f} | "
      f"So buoc grad_norm > {max_grad_norm_used}: {n_clipped}/{len(grad_norms)}")

# Luu history ra CSV, de xem lai/ve bieu do khac sau nay ma khong can train lai
import pandas as pd
df_history = pd.DataFrame({"step": range(1, len(losses)+1), "loss": losses, "lr": lrs, "grad_norm": grad_norms})
HISTORY_CSV = os.path.join(OUTPUT_DIR, "training_history.csv")
df_history.to_csv(HISTORY_CSV, index=False)
print(f"\nDa luu training history: {HISTORY_CSV}")


## (TÙY CHỌN) Train bằng LoRA thay vì UNet đầy đủ

Cell này ĐỘC LẬP với cell training ở trên (không đụng UNet đầy đủ đã train) — chỉ chạy đúng 1 trong 2 cell, không chạy cả 2. LoRA chỉ học ~vài triệu tham số mới (thay vì 860 triệu), phù hợp hơn với few-shot, giảm rủi ro "bão hòa sớm" đã thấy ở Loss.

⚠️ Checkpoint LoRA **KHÔNG dùng trực tiếp được** với `NullTextInverter`/`Editor` hiện tại — cần sửa thêm bên notebook đánh giá mới tải được (chưa làm, để sau nếu quyết định dùng).

In [ ]:
import datetime

CKPT_DIR_LORA = os.path.join(OUTPUT_DIR, "checkpoints", "specialized_unet_v2_lora")

# SUA: DUNG LAI DUNG dataset da tao cho lan train UNet day du (CUNG 3000 anh,
# KHONG tao moi) - de so sanh SACH giua LoRA vs UNet day du, chi khac PHUONG
# PHAP, KHONG khac SO LUONG ANH. Neu bien "dataset" (tu cell train UNet day du)
# van con trong bo nho (cung phien), dung lai luon - neu khong (phien moi),
# tu tao lai TU DUNG EXPANDED_LABELS_CSV (van tro toi cung file 3000 anh).
if "dataset" in dir() and len(dataset) == n_actual:
    dataset_lora = dataset
    print(f"Dung LAI dataset da co san ({len(dataset_lora)} anh) - khong tao moi.")
else:
    dataset_lora = FFHQAgingDataset(FFHQ_FULL_DIR, EXPANDED_LABELS_CSV)
    print(f"Tao MOI dataset tu EXPANDED_LABELS_CSV ({len(dataset_lora)} anh).")

print(f"\nBat dau train Module 1 BANG LoRA voi {len(dataset_lora)} anh, {train_steps_new} buoc...")
print(f"Checkpoint (LoRA, nho) se luu vao: {CKPT_DIR_LORA}\n")

specializer_lora = Specializer(
    pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5",
    lr=2e-6,                           # Giảm từ 5e-6 xuống 2e-6 cho training 512x512
    betas=(0.9, 0.999),
    train_steps=train_steps_new,
    batch_size=1,                      # ĐỔI THÀNH 1 để giải phóng VRAM trên T4
    gradient_accumulation_steps=2,     # Tích lũy 2 bước để giữ effective batch size = 2
    use_lr_decay=True,
    use_warmup=True,
    warmup_ratio=0.1,
    use_gradient_clipping=True,
    max_grad_norm=1.0,
    use_lora=True,        # SUA: BAT LoRA
    lora_rank=8,
)
t_start = datetime.datetime.now()
train_history_lora = specializer_lora.train(dataset_lora)
t_end = datetime.datetime.now()

losses_lora = train_history_lora["losses"]

specializer_lora.save_checkpoint(CKPT_DIR_LORA)

del specializer_lora
torch.cuda.empty_cache()

print(f"\nHoan tat training LoRA. Thoi gian: {t_end - t_start}")
print(f"Loss dau: {losses_lora[0]:.4f} | Loss cuoi: {losses_lora[-1]:.4f} | "
      f"Loss thap nhat: {min(losses_lora):.4f}")


## Biểu đồ đầy đủ — Loss, Learning Rate, Gradient Norm

3 biểu đồ con, giúp xác nhận trực quan Warmup/Decay/Gradient Clipping có hoạt động đúng như thiết kế không.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def smooth(values, window=20):
    if len(values) < window:
        return values
    return np.convolve(values, np.ones(window)/window, mode="valid")

fig, axes = plt.subplots(3, 1, figsize=(11, 12), sharex=False)

# ----- Bieu do 1: Loss (raw + smoothed) -----
ax = axes[0]
steps = range(1, len(losses)+1)
ax.plot(steps, losses, color="#93c5fd", linewidth=0.8, label="Loss (raw)")
smoothed = smooth(losses, window=20)
smoothed_steps = range(20, len(losses)+1)
ax.plot(smoothed_steps, smoothed, color="#2563eb", linewidth=2, label="Loss (smooth window=20)")
ax.set_title(f"FADING Specialization — Training Loss ({len(losses)} bước)")
ax.set_ylabel("MSE Loss")
ax.legend()
ax.grid(alpha=0.3)

# ----- Bieu do 2: Learning Rate -----
ax = axes[1]
ax.plot(steps, lrs, color="#16a34a", linewidth=1.5)
warmup_end = int(0.1 * len(losses))  # khop voi warmup_ratio=0.1 da dung
ax.axvline(warmup_end, color="gray", linestyle="--", alpha=0.6, label=f"Hết Warmup (bước {warmup_end})")
ax.set_title("Learning Rate theo bước — xác nhận Warmup rồi Decay")
ax.set_ylabel("Learning Rate")
ax.legend()
ax.grid(alpha=0.3)

# ----- Bieu do 3: Gradient Norm -----
ax = axes[2]
ax.plot(steps, grad_norms, color="#ea580c", linewidth=0.8, alpha=0.7)
ax.axhline(max_grad_norm_used, color="red", linestyle="--", alpha=0.6,
           label=f"Ngưỡng Clipping ({max_grad_norm_used})")
ax.set_title(f"Gradient Norm theo bước — {n_clipped}/{len(grad_norms)} bước bị clip")
ax.set_xlabel("Training Step")
ax.set_ylabel("Gradient Norm")
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
CHART_PATH = os.path.join(OUTPUT_DIR, "training_full_report.png")
plt.savefig(CHART_PATH, dpi=120)
plt.show()
print(f"\nDa luu bieu do: {CHART_PATH}")


## Nén checkpoint mới + tạo link tải về

In [ ]:
import zipfile
from IPython.display import FileLink, display

ZIP_CKPT_PATH = "/kaggle/working/specialized_unet_v2.zip"

# SUA: nen ca checkpoint LAN log/history/bieu do (khong chi rieng checkpoint nhu truoc),
# de co the xem lai qua trinh train ma khong can chay lai.
with zipfile.ZipFile(ZIP_CKPT_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(CKPT_DIR_NEW):
        for fname in files:
            fpath = os.path.join(root, fname)
            arcname = os.path.join("checkpoint", os.path.relpath(fpath, CKPT_DIR_NEW))
            zf.write(fpath, arcname=arcname)

    for extra_file in ["training_history.csv", "training_full_report.png", "training_session_log.txt"]:
        extra_path = os.path.join(OUTPUT_DIR, extra_file)
        if os.path.isfile(extra_path):
            zf.write(extra_path, arcname=extra_file)

zip_size = os.path.getsize(ZIP_CKPT_PATH) / (1024*1024)
print(f"Da nen checkpoint + log + bieu do: {ZIP_CKPT_PATH} ({zip_size:.1f} MB)")
print(">>> BAM VAO LINK DUOI DAY DE TAI VE:")
display(FileLink("specialized_unet_v2.zip"))
